# TRACS LLM Benchmark Analysis
Compares all models (GPT, Mistral, Zephyr, Llama-3.1-8B, Llama-4-Scout, Gemini-3-Flash, Claude) on the four TRACS test splits.

**Metrics:** telescope classification (accuracy, macro-F1, per-class F1, confusion matrix) and usage flag classification (science / instrumentation / mention / not_telescope — binary F1 per flag + micro/macro).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)

try:
    from datasets import load_dataset
    HF_OK = True
except ImportError:
    HF_OK = False
    print('WARNING: datasets not installed — cannot load ground truth')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Run from the baseline directory, or set TRACS_BASE_DIR.
BASE = Path(os.environ.get('TRACS_BASE_DIR', '.')).expanduser().resolve()
OLD  = BASE / 'tracs_old_results'
OUT  = BASE / 'analysis_outputs'
OUT.mkdir(exist_ok=True)

SPLITS = ['NONE_reference_test', 'JWST_reference_test', 'CHANDRA_reference_test', 'HST_reference_test']
TELESCOPE_CLASSES = ['CHANDRA', 'JWST', 'HST', 'None']
USAGE_FLAGS = ['science', 'instrumentation', 'mention', 'not_telescope']
print('Setup complete.')

In [ ]:
# ── Model registry ────────────────────────────────────────────────────────────
MODELS = {
    'GPT': {
        'NONE_reference_test':    OLD / 'gpt_tracs/None_llm_submission.csv',
        'JWST_reference_test':    OLD / 'gpt_tracs/JWST_llm_submission.csv',
        'CHANDRA_reference_test': OLD / 'gpt_tracs/CHANDRA_tracs_llm_submission.csv',
        'HST_reference_test':     OLD / 'gpt_tracs/HST_tracs_llm_submission.csv',
    },
    'Llama-3.1-8B': {
        'NONE_reference_test':    OLD / 'llamma3.18b/llama3.18b_NONE_reference_test.csv',
        'JWST_reference_test':    OLD / 'llamma3.18b/llama3.18b_JWST_reference_test.csv',
        'CHANDRA_reference_test': OLD / 'llamma3.18b/llama3.18b_CHANDRA_reference_test.csv',
        'HST_reference_test':     OLD / 'llamma3.18b/llama3.18b_HST_reference_test.csv',
    },
    'Mistral': {
        'NONE_reference_test':    OLD / 'mistral_tracs/mistral_NONE_reference_test.csv',
        'JWST_reference_test':    OLD / 'mistral_tracs/mistral_JWST_reference_test.csv',
        'CHANDRA_reference_test': OLD / 'mistral_tracs/mistral_pred_CHANDRA_reference_test.csv',
        'HST_reference_test':     OLD / 'mistral_tracs/mistral_HST_reference_test.csv',
    },
    'Zephyr': {
        'NONE_reference_test':    OLD / 'zephyr_tracs/zephyr_NONE_reference_test.csv',
        'JWST_reference_test':    OLD / 'zephyr_tracs/zephyr_JWST_reference_test.csv',
        'CHANDRA_reference_test': OLD / 'zephyr_tracs/zephyr_CHANDRA_reference_test.csv',
        'HST_reference_test':     OLD / 'zephyr_tracs/zephyr_HST_reference_test.csv',
    },
    'Llama-4-Scout': {
        'NONE_reference_test':    BASE / 'llama4_tracs_outputs/llama4_NONE_reference_test.csv',
        'JWST_reference_test':    BASE / 'llama4_tracs_outputs/llama4_JWST_reference_test.csv',
        'CHANDRA_reference_test': BASE / 'llama4_tracs_outputs/llama4_CHANDRA_reference_test.csv',
        'HST_reference_test':     BASE / 'llama4_tracs_outputs/llama4_HST_reference_test.csv',
    },
    'Gemini-3-Flash': {
        'NONE_reference_test':    BASE / 'gemini_tracs_outputs/gemini_NONE_reference_test.csv',
        'JWST_reference_test':    BASE / 'gemini_tracs_outputs/gemini_JWST_reference_test.csv',
        'CHANDRA_reference_test': BASE / 'gemini_tracs_outputs/gemini_CHANDRA_reference_test.csv',
        'HST_reference_test':     BASE / 'gemini_tracs_outputs/gemini_HST_reference_test.csv',
    },
}

for model, splits in MODELS.items():
    missing = [s for s, p in splits.items() if not Path(p).exists()]
    ok      = [s for s, p in splits.items() if Path(p).exists()]
    status  = '✓' if not missing else f'MISSING: {missing}'
    print(f'{model:20s} {len(ok)}/4 splits  {status}')

In [ ]:
# ── Normalisation helpers ─────────────────────────────────────────────────────

def norm_bool(x):
    if isinstance(x, (bool, np.bool_)): return bool(x)
    if x is None or (isinstance(x, float) and np.isnan(x)): return False
    s = str(x).strip().lower()
    return s in {'true', 't', '1', 'yes', 'y'}

def norm_tel(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return 'None'
    s = str(x).strip().upper()
    return s if s in {'CHANDRA', 'JWST', 'HST'} else 'None'

def load_preds(path):
    df = pd.read_csv(path, dtype=str)
    df = df[df['Id'] != 'Id'].copy()          # drop duplicate header rows
    df['Id'] = df['Id'].str.strip()
    df['telescope'] = df['telescope'].map(norm_tel)
    for c in USAGE_FLAGS:
        if c in df.columns:
            df[c] = df[c].map(norm_bool)
        else:
            df[c] = False
    return df[['Id', 'telescope'] + USAGE_FLAGS]

_GT_CACHE = {}
def load_gt(split):
    if split not in _GT_CACHE:
        ds = load_dataset('adsabs/TRACS', split=split)
        gt = ds.to_pandas()
        if 'Id' not in gt.columns:
            gt['Id'] = gt['bibcode']
        gt['Id'] = gt['Id'].astype(str).str.strip()
        gt['telescope'] = gt['telescope'].map(norm_tel)
        for c in USAGE_FLAGS:
            gt[c] = gt[c].map(norm_bool)
        _GT_CACHE[split] = gt[['Id', 'telescope'] + USAGE_FLAGS]
    return _GT_CACHE[split]

def align(gt, pred):
    m = gt.merge(pred, on='Id', suffixes=('_gt', '_pr'), how='inner')
    return m

print('Helpers defined.')

In [ ]:
# ── Compute all metrics ───────────────────────────────────────────────────────
# Rows: (model, split)
# Columns: tel_acc, tel_macro_f1, tel_micro_f1, usage_micro_f1, usage_macro_f1,
#          + per-class tel F1, per-flag usage F1

records = []   # summary rows
cms     = {}   # (model, split) -> confusion matrix array
tel_pc  = {}   # (model, split) -> per-class DataFrame
usg_pf  = {}   # (model, split) -> per-flag DataFrame

print('Loading ground truth ...')
for split in SPLITS:
    load_gt(split)
print('GT loaded.\n')

for model, split_paths in MODELS.items():
    for split in SPLITS:
        path = split_paths.get(split)
        if path is None or not Path(path).exists():
            continue
        gt   = load_gt(split)
        pred = load_preds(path)
        m    = align(gt, pred)
        if len(m) < 5:
            print(f'  SKIP {model}/{split}: only {len(m)} aligned rows')
            continue

        # ── Telescope metrics ────────────────────────────────────
        yt = m['telescope_gt'].values
        yp = m['telescope_pr'].values
        tel_acc      = accuracy_score(yt, yp)
        tel_macro_f1 = f1_score(yt, yp, labels=TELESCOPE_CLASSES, average='macro', zero_division=0)
        tel_micro_f1 = f1_score(yt, yp, labels=TELESCOPE_CLASSES, average='micro', zero_division=0)
        p_, r_, f_, s_ = precision_recall_fscore_support(yt, yp, labels=TELESCOPE_CLASSES, zero_division=0)
        cm_arr = confusion_matrix(yt, yp, labels=TELESCOPE_CLASSES)

        cms[(model, split)]    = cm_arr
        tel_pc[(model, split)] = pd.DataFrame({'class': TELESCOPE_CLASSES, 'P': p_, 'R': r_, 'F1': f_, 'support': s_})

        # ── Usage-flag metrics (multi-label binary) ──────────────
        flag_rows = []
        yt_all, yp_all = [], []
        for flag in USAGE_FLAGS:
            yf_t = m[f'{flag}_gt'].astype(int).values
            yf_p = m[f'{flag}_pr'].astype(int).values
            pf, rf, ff, _ = precision_recall_fscore_support(yf_t, yf_p, average='binary', zero_division=0)
            flag_rows.append({'flag': flag, 'P': pf, 'R': rf, 'F1': ff, 'support': int(yf_t.sum())})
            yt_all.append(yf_t);  yp_all.append(yf_p)
        usg_pf[(model, split)] = pd.DataFrame(flag_rows)
        Yt = np.stack(yt_all, 1);  Yp = np.stack(yp_all, 1)
        usg_micro = f1_score(Yt, Yp, average='micro',  zero_division=0)
        usg_macro = f1_score(Yt, Yp, average='macro',  zero_division=0)

        row = {'model': model, 'split': split, 'n': len(m),
               'tel_acc': tel_acc, 'tel_macro_f1': tel_macro_f1, 'tel_micro_f1': tel_micro_f1,
               'usg_micro_f1': usg_micro, 'usg_macro_f1': usg_macro}
        for cls, f1v in zip(TELESCOPE_CLASSES, f_):
            row[f'tel_f1_{cls}'] = f1v
        for fr in flag_rows:
            row[f'usg_f1_{fr["flag"]}'] = fr['F1']
        records.append(row)

df_all = pd.DataFrame(records)
print(f'\nComputed metrics for {len(df_all)} (model, split) combinations.')
df_all.head()

In [ ]:
# ── Leaderboard (average across splits) ──────────────────────────────────────

lb_cols = ['tel_macro_f1', 'tel_micro_f1', 'tel_acc', 'usg_macro_f1', 'usg_micro_f1']
leaderboard = (
    df_all.groupby('model')[lb_cols]
    .mean()
    .sort_values('tel_macro_f1', ascending=False)
    .reset_index()
)
leaderboard.columns = ['Model', 'Tel Macro-F1', 'Tel Micro-F1', 'Tel Acc',
                        'Usage Macro-F1', 'Usage Micro-F1']
leaderboard = leaderboard.round(3)
leaderboard.index = range(1, len(leaderboard)+1)   # rank
leaderboard.to_csv(OUT / 'leaderboard.csv')

display(leaderboard.style
    .background_gradient(cmap='RdYlGn', subset=['Tel Macro-F1','Tel Micro-F1','Tel Acc','Usage Macro-F1','Usage Micro-F1'])
    .format('{:.3f}', subset=['Tel Macro-F1','Tel Micro-F1','Tel Acc','Usage Macro-F1','Usage Micro-F1'])
    .set_caption('Leaderboard — averages across all 4 TRACS splits')
)

In [ ]:
# ── Per-split summary table ───────────────────────────────────────────────────

pivot = df_all.pivot_table(index='model', columns='split', values='tel_macro_f1').round(3)
pivot['AVG'] = pivot.mean(axis=1)
pivot = pivot.sort_values('AVG', ascending=False)
display(pivot.style
    .background_gradient(cmap='RdYlGn', axis=None)
    .set_caption('Telescope Macro-F1 per split')
    .format('{:.3f}')
)

In [ ]:
# ── Heatmap: model × split — telescope macro-F1 ───────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, title in zip(
    axes,
    ['tel_macro_f1', 'usg_macro_f1'],
    ['Telescope Macro-F1', 'Usage Flags Macro-F1'],
):
    piv = df_all.pivot_table(index='model', columns='split', values=metric).round(3)
    piv = piv.reindex(leaderboard['Model'].tolist())  # rank order
    piv.columns = [c.replace('_reference_test', '') for c in piv.columns]
    sns.heatmap(piv, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=.5, cbar_kws={'shrink': .8})
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Model Performance Heatmap (all splits)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(OUT / 'heatmap_all_splits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices — one figure per split, subplots per model ─────────────

MODELS_IN_ORDER = leaderboard['Model'].tolist()

for split in SPLITS:
    models_with_cm = [m for m in MODELS_IN_ORDER if (m, split) in cms]
    n = len(models_with_cm)
    if n == 0: continue

    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4.2))
    axes = np.array(axes).flatten()

    for ax, model in zip(axes, models_with_cm):
        cm_arr = cms[(model, split)]
        # normalise by true-class totals for readability
        row_sums = cm_arr.sum(axis=1, keepdims=True).clip(min=1)
        cm_norm  = cm_arr / row_sums
        sns.heatmap(cm_norm, ax=ax,
                    annot=np.array([[f'{v:.0%}\n({c})' for v, c in zip(row_n, row_c)]
                                    for row_n, row_c in zip(cm_norm, cm_arr)]),
                    fmt='', cmap='Blues', vmin=0, vmax=1,
                    xticklabels=TELESCOPE_CLASSES, yticklabels=TELESCOPE_CLASSES,
                    linewidths=.5, cbar=False)
        ax.set_title(model, fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    for ax in axes[n:]:
        ax.set_visible(False)

    split_short = split.replace('_reference_test', '')
    fig.suptitle(f'Telescope Confusion Matrices — {split_short}', fontsize=14, fontweight='bold', y=1.01)
    fig.tight_layout()
    fig.savefig(OUT / f'cm_{split_short}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

In [ ]:
# ── Per-class telescope F1 — grouped bar per split ────────────────────────────

for split in SPLITS:
    rows = df_all[df_all['split'] == split]
    if rows.empty: continue

    f1_cols = [f'tel_f1_{c}' for c in TELESCOPE_CLASSES]
    plot_df = rows.set_index('model')[f1_cols].rename(columns=lambda c: c.replace('tel_f1_', ''))
    plot_df = plot_df.reindex([m for m in MODELS_IN_ORDER if m in plot_df.index])

    split_short = split.replace('_reference_test', '')
    ax = plot_df.plot(kind='bar', figsize=(10, 4), width=0.75,
                      colormap='tab10', edgecolor='white')
    ax.set_title(f'Per-class Telescope F1 — {split_short}', fontweight='bold')
    ax.set_ylabel('F1 score')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Telescope class', bbox_to_anchor=(1, 1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    plt.tight_layout()
    plt.savefig(OUT / f'tel_perclass_{split_short}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Usage flag F1 — grouped bar per split ────────────────────────────────────

for split in SPLITS:
    rows = df_all[df_all['split'] == split]
    if rows.empty: continue

    f1_cols = [f'usg_f1_{f}' for f in USAGE_FLAGS]
    plot_df = rows.set_index('model')[f1_cols].rename(columns=lambda c: c.replace('usg_f1_', ''))
    plot_df = plot_df.reindex([m for m in MODELS_IN_ORDER if m in plot_df.index])

    split_short = split.replace('_reference_test', '')
    ax = plot_df.plot(kind='bar', figsize=(10, 4), width=0.75,
                      colormap='Set2', edgecolor='white')
    ax.set_title(f'Usage Flag F1 — {split_short}', fontweight='bold')
    ax.set_ylabel('F1 score')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Flag', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(OUT / f'usg_perflag_{split_short}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Overall model comparison (avg across splits) ──────────────────────────────

avg = df_all.groupby('model').agg(
    tel_macro_f1   = ('tel_macro_f1',  'mean'),
    tel_acc        = ('tel_acc',        'mean'),
    usg_macro_f1   = ('usg_macro_f1',  'mean'),
).reset_index().sort_values('tel_macro_f1', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
metrics = [('tel_macro_f1', 'Telescope\nMacro-F1', '#4C72B0'),
           ('tel_acc',      'Telescope\nAccuracy',  '#55A868'),
           ('usg_macro_f1', 'Usage Flags\nMacro-F1','#C44E52')]

for ax, (col, label, color) in zip(axes, metrics):
    bars = ax.barh(avg['model'], avg[col], color=color, edgecolor='white')
    for bar, val in zip(bars, avg[col]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Score')
    ax.axvline(0.5, color='gray', linewidth=0.8, linestyle='--')

fig.suptitle('Model Comparison (avg over 4 splits)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Radar / spider chart — avg per-class F1 ───────────────────────────────────

from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

cats = TELESCOPE_CLASSES + USAGE_FLAGS
N = len(cats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette('tab10', n_colors=len(MODELS_IN_ORDER))
handles = []

for model, color in zip(MODELS_IN_ORDER, colors):
    rows = df_all[df_all['model'] == model]
    if rows.empty: continue
    vals = []
    for cls in TELESCOPE_CLASSES:
        vals.append(rows[f'tel_f1_{cls}'].mean())
    for flag in USAGE_FLAGS:
        vals.append(rows[f'usg_f1_{flag}'].mean())
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linewidth=2, linestyle='solid')
    ax.fill(angles, vals, color=color, alpha=0.1)
    handles.append(mpatches.Patch(color=color, label=model))

ax.set_thetagrids(np.degrees(angles[:-1]), cats)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25','0.5','0.75','1.0'], fontsize=8)
ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.35, 1.1))
ax.set_title('Per-class F1 Radar (avg over splits)', fontweight='bold', pad=20)
fig.tight_layout()
fig.savefig(OUT / 'radar_perclass.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error analysis — find hard rows (models disagree or all wrong) ────────────
# Focus on JWST split as the most meaningful

TARGET_SPLIT = 'JWST_reference_test'
gt = load_gt(TARGET_SPLIT)

vote_frames = []
for model, split_paths in MODELS.items():
    path = split_paths.get(TARGET_SPLIT)
    if path is None or not Path(path).exists(): continue
    pred = load_preds(path)
    m = gt.merge(pred, on='Id', suffixes=('_gt', '_pr'), how='inner')
    if len(m) < 5: continue
    m = m.rename(columns={f'telescope_pr': f'tel_{model}',
                           **{f'{f}_pr': f'{f}_{model}' for f in USAGE_FLAGS}})
    vote_frames.append(m[['Id', f'tel_{model}']])

if vote_frames:
    from functools import reduce
    votes = reduce(lambda a, b: a.merge(b, on='Id', how='outer'), vote_frames)
    votes = votes.merge(gt[['Id','telescope']], on='Id', how='left')

    tel_cols = [c for c in votes.columns if c.startswith('tel_') and c != 'telescope']
    votes['n_models'] = votes[tel_cols].notna().sum(axis=1)
    votes['n_correct'] = votes.apply(
        lambda r: sum(r[c] == r['telescope'] for c in tel_cols if pd.notna(r[c])), axis=1)
    votes['pct_correct'] = votes['n_correct'] / votes['n_models'].clip(lower=1)

    # Hardest rows: all models got wrong
    hard = votes[votes['pct_correct'] == 0].sort_values('n_models', ascending=False)
    easy = votes[votes['pct_correct'] == 1].sort_values('n_models', ascending=False)

    print(f'Total rows (JWST split): {len(votes)}')
    print(f'Rows ALL models got right: {len(easy)} ({len(easy)/len(votes):.1%})')
    print(f'Rows ALL models got wrong: {len(hard)} ({len(hard)/len(votes):.1%})')
    print()

    # Distribution of % correct per row
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(votes['pct_correct'], bins=20, edgecolor='white', color='steelblue')
    ax.set_xlabel('Fraction of models correct on this row')
    ax.set_ylabel('# papers')
    ax.set_title(f'Difficulty distribution — {TARGET_SPLIT.replace("_reference_test","")}', fontweight='bold')
    fig.tight_layout()
    fig.savefig(OUT / 'difficulty_dist_JWST.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Sample of hardest papers (all models wrong):')
    display(hard[['Id', 'telescope'] + tel_cols].head(10))

In [ ]:
# ── Model agreement matrix ────────────────────────────────────────────────────
if vote_frames:
    model_names_in_votes = [c.replace('tel_', '') for c in tel_cols]
    agree = np.full((len(tel_cols), len(tel_cols)), np.nan)
    for i, ci in enumerate(tel_cols):
        for j, cj in enumerate(tel_cols):
            sub = votes[[ci, cj]].dropna()
            if len(sub):
                agree[i, j] = float((sub.iloc[:, 0] == sub.iloc[:, 1]).mean())

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(agree, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=model_names_in_votes,
                yticklabels=model_names_in_votes,
                vmin=0, vmax=1, linewidths=.5, ax=ax)
    ax.set_title('Model Agreement on Telescope (JWST split)\nFraction of rows where both models agree',
                 fontweight='bold')
    fig.tight_layout()
    fig.savefig(OUT / 'model_agreement_JWST.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Save full metrics table ───────────────────────────────────────────────────

df_all.to_csv(OUT / 'all_metrics.csv', index=False)
print(f'All outputs saved to {OUT}')
print(f'Files: {sorted(p.name for p in OUT.iterdir())}')